# 08 — Validation Benchmark

Loads the real VAL report (`report.json`, `records.csv`, `report.csv`) produced by `ValidationRunner.run()` (`src/validation/validate.py` -> `python run.py --mode validate`), reproduces the README's 9-stage metrics table, and adds image-level triage: which images are fully correct, which were only saved by plugin fusion, which have the wrong count, and which have the right count but wrong identity.

**Schema note:** every field referenced below is copied verbatim from `src/validation/evaluator.py` (`Evaluator._build_records`, `_per_image_summary`, `_evaluate_plugins`, ...) — not guessed from the report's shape. `metrics.py` (the `compute_metrics` formulas) was not provided, so metric lookups use `.get(..., "N/A")` defensively; §3 prints the report's actual top-level and nested keys before any table is built, so a stale assumption fails loudly instead of silently.

Covered in this notebook:

1. Setup and helpers
2. Config overview
3. Load `report.json` / `records.csv` / `report.csv` (or run VAL if missing — gated, expensive)
4. Stage-by-stage metrics tables (Detection -> ... -> End-to-End)
5. Per-product breakdown
6. Per-image distributions and latency breakdown
7. **Image triage:** fully correct / corrected by plugin / wrong count / right count but wrong identity
8. Plugin flip analysis (aggregate + optional per-crop drill-down)
9. Confusable-pair error breakdown
10. Latency vs. per-image correctness
11. Optional: re-run VAL with a patched config (gated)
12. Summary

## 1. Environment setup

In [ ]:
import sys, os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "configs" / "config.yaml").is_file()

In [ ]:
import json

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.core.config import load_config

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_columns", 60)
%matplotlib inline


def show_image_bgr(ax, image_array_bgr, title=""):
    ax.imshow(cv2.cvtColor(image_array_bgr, cv2.COLOR_BGR2RGB))
    ax.set_title(title, fontsize=9)
    ax.axis("off")


def to_bool(series):
    """Normalizes a records.csv column to real booleans.

    csv.DictWriter serializes Python bool via str() ('True'/'False'); pandas
    often infers these as bool automatically on read, but not always
    (e.g. mixed with NaN for the synthetic missed-GT rows) -- this handles
    both cases without assuming which one happened.
    """
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower() == "true"


def to_id_str(series):
    """Normalizes a product-id column back to string, preserving NaN as None.

    product_id is str(...) throughout the codebase (GroundTruthObject,
    RetrievalCandidate, ...), but pandas can infer an all-numeric-looking
    CSV column as int64/float64 on read.
    """
    return series.where(series.isna(), series.astype(str).str.replace(r"\.0$", "", regex=True))

## 2. Config overview

In [ ]:
config = load_config()

output_dir = config.resolve_path(config.paths.output_dir)
print(f"Output dir:              {output_dir}")
print(f"report_json_filename:    {config.validation.report_json_filename}")
print(f"report_csv_filename:     {config.validation.report_csv_filename}")
print(f"records_filename:        {config.validation.records_filename}")
print(f"annotated_images_dirname:{config.validation.annotated_images_dirname}")
print(f"\niou_match_threshold:     {config.validation.iou_match_threshold}")
print(f"top_k:                   {config.validation.top_k}")
print(f"\nconfusable_pairs:        {config.rerank.confusable_pairs}")

## 3. Load `report.json` / `records.csv` / `report.csv`

Uses the exact filenames from `configs/config.yaml -> validation.*`, matching what `ValidationRunner._write_reports` actually wrote — not hardcoded names like `demo_report.json`. If they're missing, `RUN_VALIDATION_IF_MISSING` gates an optional full `ValidationRunner.run()` call, which re-runs `InventoryPipeline` over the entire benchmark and can take a long time (README reports ~49s/image mean end-to-end latency).

In [ ]:
json_path = output_dir / config.validation.report_json_filename
records_path = output_dir / config.validation.records_filename
report_csv_path = output_dir / config.validation.report_csv_filename

RUN_VALIDATION_IF_MISSING = False  # set True to actually re-run VAL over the full benchmark (slow)

if not json_path.is_file() and RUN_VALIDATION_IF_MISSING:
    from src.validation.validate import ValidationRunner
    benchmark_dir = config.resolve_path(config.paths.benchmark_dir)
    print(f"Running ValidationRunner over '{benchmark_dir}' — this reruns the full pipeline per image...")
    ValidationRunner(config).run(str(benchmark_dir))

assert json_path.is_file(), (
    f"report not found at {json_path}. Run `python run.py --mode validate --benchmark-dir data/benchmark/` "
    "first, or set RUN_VALIDATION_IF_MISSING=True above."
)

with open(json_path, "r", encoding="utf-8") as f:
    report = json.load(f)

records_df = pd.read_csv(records_path) if records_path.is_file() else pd.DataFrame()
report_csv_df = pd.read_csv(report_csv_path) if report_csv_path.is_file() else pd.DataFrame()

print(f"report.json loaded.  records.csv: {records_df.shape}  report.csv: {report_csv_df.shape}")

Normalize dtypes and inspect the **real** top-level schema before building anything on top of it:

In [ ]:
print("report.json top-level keys:", list(report.keys()))
print("\nreport['detection'] keys:      ", list(report["detection"].keys()) if isinstance(report.get("detection"), dict) else report.get("detection"))
print("report['plugins'] keys:         ", list(report["plugins"].keys()) if isinstance(report.get("plugins"), dict) else report.get("plugins"))
if isinstance(report.get("plugins"), dict) and report["plugins"]:
    first_plugin = next(iter(report["plugins"]))
    print(f"report['plugins']['{first_plugin}'] keys:", list(report["plugins"][first_plugin].keys()))

if not records_df.empty:
    print("\nrecords.csv columns:", list(records_df.columns))
    for bool_col in ["gt_matched", "overlap_flagged", "refinement_triggered", "refinement_used_fallback",
                       "changed_by_plugin", "correct"]:
        if bool_col in records_df.columns:
            records_df[bool_col] = to_bool(records_df[bool_col])
    for id_col in ["gt_product_id", "retrieval_top1_product_id", "final_product_id"]:
        if id_col in records_df.columns:
            records_df[id_col] = to_id_str(records_df[id_col])

## 4. Stage-by-stage metrics tables

Each table only includes fields directly confirmed in `evaluator.py`'s `.update({...})` calls for that stage (never a guessed metric name); the raw `precision`/`recall`/`f1`/`mean_iou`-style keys from `compute_metrics()` are read defensively via `.get()`.

In [ ]:
def stage_table(stage_name, fields):
    stage = report.get(stage_name)
    if stage == "skipped_by_config":
        print(f"[{stage_name}] skipped_by_config")
        return pd.DataFrame()
    if not isinstance(stage, dict):
        print(f"[{stage_name}] unavailable")
        return pd.DataFrame()
    rows = [{"metric": name, "value": stage.get(name, "N/A")} for name in fields]
    return pd.DataFrame(rows)


print("=== Detection ===")
display(stage_table("detection", [
    "precision", "recall", "f1", "mean_iou", "gt_objects", "predicted_objects",
    "true_positive", "false_positive", "false_negative",
    "confidence_mean", "confidence_min", "confidence_max",
]))

In [ ]:
print("=== Cropping & Overlap ===")
display(stage_table("cropping", ["valid_rate", "mean_iou", "valid_crop_count", "invalid_crop_count", "crop_to_gt_matching_rate"]))
display(stage_table("overlap", ["precision", "recall", "f1", "true_positive", "false_positive", "false_negative",
                                  "sent_to_refiner_count", "correctly_left_unrefined_count"]))

In [ ]:
print("=== Segmentation ===")
display(stage_table("segmentation", [
    "iou_improvement", "improved_rate", "refinement_triggered_count", "refinement_success_count",
    "refinement_failure_count", "degraded_count", "unchanged_count",
]))

In [ ]:
print("=== Retrieval ===")
display(stage_table("retrieval", [
    "top1_accuracy", "topk_accuracy", "mrr", "mean_rank", "recall_at_k",
    "evaluated_crops", "mean_top1_similarity",
]))

In [ ]:
print("=== Decision ===")
display(stage_table("decision", [
    "accuracy", "precision", "recall", "f1",
    "correct_accepted", "incorrect_accepted", "false_rejection",
    "uncertain_case_count", "ambiguous_case_count",
]))

decision_cm = report.get("decision", {}).get("confusion_matrix") if isinstance(report.get("decision"), dict) else None
if isinstance(decision_cm, dict) and decision_cm:
    print("\nDecision confusion_matrix ({actual: {predicted: count}}):")
    display(pd.DataFrame(decision_cm).fillna(0).astype(int))

In [ ]:
print("=== Plugins (per plugin) ===")
plugins_report = report.get("plugins", {})
if isinstance(plugins_report, dict):
    plugin_rows = []
    for name, stats in plugins_report.items():
        plugin_rows.append({
            "plugin": name,
            "eligible": stats.get("eligible"), "executed": stats.get("executed"), "skipped": stats.get("skipped"),
            "success": stats.get("success"), "failure": stats.get("failure"),
            "identified_total": stats.get("identified_total"), "identified_correct": stats.get("identified_correct"),
            "corrected": stats.get("corrected"), "degraded": stats.get("degraded"), "influenced": stats.get("influenced"),
        })
    plugins_df = pd.DataFrame(plugin_rows)
    display(plugins_df)
else:
    plugins_df = pd.DataFrame()
    print("skipped_by_config")

In [ ]:
print("=== Fusion ===")
display(stage_table("fusion", [
    "accuracy_before", "accuracy_after", "improvement", "correction_rate",
    "corrected_cases", "newly_incorrect_cases", "unchanged_cases", "evaluated_cases", "mean_fusion_latency_ms",
]))

print("\n=== End-to-End ===")
display(stage_table("end_to_end", [
    "precision", "recall", "f1", "true_positive", "false_positive", "false_negative",
    "product_count_accuracy", "mean_latency_ms",
]))

end_to_end_cm = report.get("end_to_end", {}).get("confusion_matrix") if isinstance(report.get("end_to_end"), dict) else None
if isinstance(end_to_end_cm, dict) and end_to_end_cm:
    print("\nEnd-to-End confusion_matrix ({actual: {predicted: count}}):")
    cm_df = pd.DataFrame(end_to_end_cm).fillna(0).astype(int)
    display(cm_df)

    fig, ax = plt.subplots(figsize=(max(6, len(cm_df.columns) * 0.7), max(5, len(cm_df.index) * 0.6)))
    im = ax.imshow(cm_df.values, cmap="Blues")
    ax.set_xticks(range(len(cm_df.columns)))
    ax.set_xticklabels(cm_df.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(cm_df.index)))
    ax.set_yticklabels(cm_df.index)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Ground truth")
    ax.set_title("End-to-End confusion matrix")
    for i in range(len(cm_df.index)):
        for j in range(len(cm_df.columns)):
            ax.text(j, i, int(cm_df.values[i, j]), ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

## 5. Per-product breakdown

In [ ]:
per_product = pd.DataFrame(report.get("end_to_end", {}).get("per_product", []))
if not per_product.empty:
    display(per_product.sort_values("recall", ascending=False))

    x = np.arange(len(per_product))
    fig, ax = plt.subplots(figsize=(max(8, len(per_product) * 0.6), 4.5))
    ax.bar(x - 0.2, per_product["precision"], width=0.4, label="Precision")
    ax.bar(x + 0.2, per_product["recall"], width=0.4, label="Recall")
    ax.set_xticks(x)
    ax.set_xticklabels(per_product["product_id"], rotation=45, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.set_title("End-to-End precision/recall by product")
    plt.tight_layout()
    plt.show()
else:
    print("No per-product data available.")

## 6. Per-image distributions and latency breakdown

In [ ]:
per_image_df = pd.DataFrame(report.get("per_image", []))
if not per_image_df.empty:
    per_image_df["count_error"] = per_image_df["predicted_count"] - per_image_df["ground_truth_count"]
    display(per_image_df.describe())

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].hist(per_image_df["processing_time_ms"] / 1000.0, bins=12, color="#4C72B0")
    axes[0].set_xlabel("Processing time (s)")
    axes[0].set_ylabel("Number of images")
    axes[0].set_title("End-to-end latency distribution")

    axes[1].bar(range(len(per_image_df)), per_image_df["count_error"], color="#DD8452")
    axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
    axes[1].set_xlabel("Image index")
    axes[1].set_ylabel("predicted_count - ground_truth_count")
    axes[1].set_title("Object count error by image")
    plt.tight_layout()
    plt.show()
else:
    print("No per-image data available.")

In [ ]:
latency = report.get("latency", {})
if latency:
    lat_rows = [{"stage": name, "mean_ms": v["mean"], "min_ms": v["min"], "max_ms": v["max"]} for name, v in latency.items()]
    lat_df = pd.DataFrame(lat_rows).sort_values("mean_ms", ascending=False)
    display(lat_df)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(lat_df["stage"], lat_df["mean_ms"], color="#55A868")
    ax.set_xlabel("Mean latency (ms)")
    ax.set_title("Mean latency by pipeline stage")
    plt.tight_layout()
    plt.show()
else:
    print("No latency data available.")

## 7. Image triage

Four buckets, built purely from `records.csv` + `per_image` (no pipeline re-run needed):

- **Fully correct end-to-end** — every row for that image has `correct == True` (this already excludes false positives and missed-GT rows, since both are constructed to have `correct == False` by `_build_records`).
- **Corrected by plugin fusion** — a crop counts as "saved by fusion" when it ended up `correct == True` but demonstrably was **not** already correct before reranking. Derived without needing the pre-fusion `product_id` (not stored in `records.csv`) using only confirmed columns:
    - `decision_status_before != "accepted"` (wasn't even accepted pre-fusion), **or**
    - `changed_by_plugin == True` **and** the crop is now correct (if the winning product changed and the new one matches GT, the old one — by definition different — could not have matched GT too).
- **Wrong count** — `predicted_count != ground_truth_count` (from `per_image`, which already reflects `InventoryPipeline`'s real inclusion rule: `accepted` **and** `uncertain` items with a resolved `product_id` both count, only `rejected`/`product_id=None` are excluded).
- **Right count, wrong identification** — `predicted_count == ground_truth_count` but not fully correct (at least one product misidentified even though the count matches).

In [ ]:
if not records_df.empty and not per_image_df.empty:
    def plugin_ran(row):
        return isinstance(row["plugins_executed"], str) and row["plugins_executed"].strip() != ""

    records_df["_plugin_ran"] = records_df.apply(plugin_ran, axis=1) if "plugins_executed" in records_df.columns else False
    records_df["_corrected_by_plugin"] = (
        records_df["correct"]
        & records_df["_plugin_ran"]
        & ((records_df["decision_status_before"] != "accepted") | records_df["changed_by_plugin"])
    )

    per_image_agg = records_df.groupby("image_key").agg(
        all_correct=("correct", "all"),
        any_corrected_by_plugin=("_corrected_by_plugin", "any"),
        n_rows=("correct", "size"),
    ).reset_index()

    triage_df = per_image_agg.merge(per_image_df, on="image_key", how="left")

    bucket_fully_correct = triage_df[triage_df["all_correct"]]
    bucket_corrected_by_plugin = triage_df[triage_df["all_correct"] & triage_df["any_corrected_by_plugin"]]
    bucket_wrong_count = triage_df[triage_df["predicted_count"] != triage_df["ground_truth_count"]]
    bucket_right_count_wrong_id = triage_df[
        (triage_df["predicted_count"] == triage_df["ground_truth_count"]) & (~triage_df["all_correct"])
    ]

    print(f"Fully correct end-to-end:        {len(bucket_fully_correct)} / {len(triage_df)} image(s)")
    print(f"  ...of which corrected by plugin: {len(bucket_corrected_by_plugin)}")
    print(f"Wrong predicted count:            {len(bucket_wrong_count)} / {len(triage_df)} image(s)")
    print(f"Right count, wrong identification:{len(bucket_right_count_wrong_id)} / {len(triage_df)} image(s)")
else:
    triage_df = pd.DataFrame()
    print("records.csv or per_image data unavailable — skip.")

In [ ]:
if not triage_df.empty:
    print("\n--- Fully correct end-to-end ---")
    display(bucket_fully_correct[["image_key", "predicted_count", "ground_truth_count", "processing_time_ms"]])

In [ ]:
if not triage_df.empty:
    print("\n--- Corrected by plugin fusion (subset of fully-correct) ---")
    display(bucket_corrected_by_plugin[["image_key", "predicted_count", "ground_truth_count"]])

In [ ]:
if not triage_df.empty:
    print("\n--- Wrong predicted count ---")
    display(bucket_wrong_count[["image_key", "predicted_count", "ground_truth_count", "count_error" if "count_error" in bucket_wrong_count.columns else "predicted_count"]])

In [ ]:
if not triage_df.empty:
    print("\n--- Right count, wrong identification ---")
    display(bucket_right_count_wrong_id[["image_key", "predicted_count", "ground_truth_count"]])

Visual spot-check using the **real** annotated images `ValidationRunner._save_annotated_images` already wrote (green solid = TP, red solid = wrong product, orange dashed = missed GT) — no re-rendering.

In [ ]:
annotated_dir = output_dir / config.validation.annotated_images_dirname

def show_examples(bucket_df, title, n=3):
    if bucket_df.empty:
        print(f"{title}: no examples.")
        return
    sample = bucket_df.head(n)
    fig, axes = plt.subplots(1, len(sample), figsize=(4.5 * len(sample), 4.5))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        image_key_val = row["image_key"]
        img_path = annotated_dir / f"{image_key_val}.jpg"
        if img_path.is_file():
            show_image_bgr(ax, cv2.imread(str(img_path)), title=row["image_key"])
        else:
            ax.text(0.5, 0.5, "annotated image not found", ha="center", va="center")
            ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

if not triage_df.empty and annotated_dir.is_dir():
    show_examples(bucket_wrong_count, "Wrong predicted count — examples")
    show_examples(bucket_right_count_wrong_id, "Right count, wrong identification — examples")
elif not triage_df.empty:
    print(f"Annotated images directory not found at {annotated_dir} "
          "(re-run VAL with validation.save_annotated_images=true).")

## 8. Plugin flip analysis

The aggregate `corrected`/`degraded`/`influenced` counts per plugin (§4's plugin table, computed by `Evaluator._evaluate_plugins` from `rerank_debug["candidates"]` match-strength per crop — real signal, not re-derived here). Compares each plugin's raw usefulness (`success_rate`) against its actual effect on correctness (`corrected` vs `degraded`).

In [ ]:
if not plugins_df.empty:
    for name in plugins_report:
        for key in ("success_rate", "coverage_rate", "correction_rate", "degradation_rate", "influence_rate"):
            plugins_df.loc[plugins_df["plugin"] == name, key] = plugins_report[name].get(key)
    plugins_df["net_effect"] = plugins_df["corrected"] - plugins_df["degraded"]
    display(plugins_df[[
        "plugin", "eligible", "executed", "success_rate", "coverage_rate",
        "corrected", "degraded", "influenced", "correction_rate", "degradation_rate", "influence_rate", "net_effect",
    ]])

    fig, ax = plt.subplots(figsize=(8, 4.5))
    x = np.arange(len(plugins_df))
    ax.bar(x - 0.2, plugins_df["corrected"], width=0.4, label="corrected (wrong -> right)", color="#55A868")
    ax.bar(x + 0.2, plugins_df["degraded"], width=0.4, label="degraded (right -> wrong)", color="#C44E52")
    ax.set_xticks(x)
    ax.set_xticklabels(plugins_df["plugin"])
    ax.set_ylabel("Crop count")
    ax.set_title("Per-plugin correction vs. degradation")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No plugin data available.")

**Optional drill-down:** re-runs `InventoryPipeline.run_with_trace()` on one flagged image to inspect `rerank_debug` directly (same technique as `07_end_to_end_pipeline.ipynb` §8) — useful when a plugin's aggregate `degraded` count looks high and you want to see exactly which candidate it favored and why.

In [ ]:
DRILL_DOWN_IMAGE_KEY = None  # set to an image_key from the tables above, e.g. bucket_right_count_wrong_id["image_key"].iloc[0]

if DRILL_DOWN_IMAGE_KEY is not None:
    from src.core.utils import load_image_bgr
    from src.models.models import ImageData
    from src.pipeline.pipeline import InventoryPipeline

    match = records_df[records_df["image_key"] == DRILL_DOWN_IMAGE_KEY]
    source_path = Path(match["source_path"].iloc[0])

    drill_pipeline = InventoryPipeline(config)
    img_array = load_image_bgr(source_path)
    h, w = img_array.shape[:2]
    img_data = ImageData(image_id=DRILL_DOWN_IMAGE_KEY, source_path=str(source_path),
                          image_array=img_array, width=w, height=h)
    _, drill_trace = drill_pipeline.run_with_trace(img_data)

    for ct in drill_trace.crops:
        if ct.plugin_result is None:
            continue
        debug = getattr(ct.final_decision, "rerank_debug", None)
        if debug:
            print(f"crop_id={ct.crop.crop_id}  winner={ct.final_decision.product_id}  status={ct.final_decision.status}")
            display(pd.DataFrame(debug["candidates"])[
                ["product_id", "base_similarity", "barcode_boost", "ocr_boost", "color_boost", "adjusted_score"]
            ])
else:
    print("DRILL_DOWN_IMAGE_KEY not set — skip (set it to an image_key from §7's tables to inspect that image).")

## 9. Confusable-pair error breakdown

Of every misidentification (`final_product_id != gt_product_id`, both present), how many fall into a pair already listed in `config.rerank.confusable_pairs` (where the guard in `Reranker._confusable_pair_conflict` applies) vs. pairs that are **not yet configured** but are empirically common — a direct signal for whether `confusable_pairs` should be extended.

In [ ]:
if not records_df.empty:
    confusable_sets = [frozenset(map(str, pair)) for pair in config.rerank.confusable_pairs]

    misid = records_df[
        records_df["final_product_id"].notna()
        & records_df["gt_product_id"].notna()
        & (records_df["final_product_id"] != records_df["gt_product_id"])
    ].copy()

    def is_confusable(row):
        pair = frozenset({str(row["gt_product_id"]), str(row["final_product_id"])})
        return pair in confusable_sets

    misid["is_configured_confusable_pair"] = misid.apply(is_confusable, axis=1) if not misid.empty else []

    print(f"Total misidentifications: {len(misid)}")
    if not misid.empty:
        confusable_hit_count = misid["is_configured_confusable_pair"].sum()
        print(f"  ...within a configured confusable_pair: {confusable_hit_count}")

        pair_counts = (
            misid.assign(pair=misid.apply(lambda r: tuple(sorted([str(r["gt_product_id"]), str(r["final_product_id"])])), axis=1))
            ["pair"].value_counts().reset_index()
        )
        pair_counts.columns = ["gt_vs_predicted_pair", "count"]
        display(pair_counts)
else:
    print("records.csv unavailable — skip.")

## 10. Latency vs. per-image correctness

Which images are both slow and wrong — the highest-value targets for further investigation.

In [ ]:
if not records_df.empty and not per_image_df.empty:
    accuracy_per_image = records_df.groupby("image_key")["correct"].mean().rename("accuracy").reset_index()
    scatter_df = accuracy_per_image.merge(per_image_df, on="image_key", how="left")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(scatter_df["processing_time_ms"] / 1000.0, scatter_df["accuracy"], alpha=0.7, color="#4C72B0")
    ax.set_xlabel("Processing time (s)")
    ax.set_ylabel("Fraction of rows correct in this image")
    ax.set_title("Latency vs. per-image correctness")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

    worst = scatter_df.sort_values(["accuracy", "processing_time_ms"], ascending=[True, False]).head(5)
    print("Slowest + least-correct images:")
    display(worst[["image_key", "accuracy", "processing_time_ms"]])
else:
    print("records.csv or per_image data unavailable — skip.")

## 11. Optional: re-run VAL with a patched config

Gated and off by default (re-running VAL over the full benchmark is expensive). Useful for quantifying the effect of a candidate fix — e.g. `06_plugins_deep_dive.ipynb` §8 found that `ColorPlugin` never populates a top-level `confidence` key, so `color` evidence's effective boost is always `0.0` in the live pipeline. This cell lets you simulate a fix (by monkeypatching evidence at evaluation time is out of scope here — this only demonstrates the *config-sweep* pattern for tunable thresholds, e.g. comparing two `decision.similarity_threshold` values) without editing `config.yaml`.

In [ ]:
RUN_CONFIG_SWEEP = False  # set True to actually run this (slow — one full VAL pass per config variant)

if RUN_CONFIG_SWEEP:
    from src.validation.validate import ValidationRunner

    variants = {
        "baseline": config,
        "similarity_threshold_minus_0.05": config.model_copy(update={
            "decision": config.decision.model_copy(update={
                "similarity_threshold": config.decision.similarity_threshold - 0.05
            })
        }),
    }

    sweep_results = {}
    benchmark_dir = config.resolve_path(config.paths.benchmark_dir)
    for variant_name, variant_config in variants.items():
        print(f"Running VAL for variant '{variant_name}'...")
        variant_report = ValidationRunner(variant_config).run(str(benchmark_dir))
        sweep_results[variant_name] = {
            "end_to_end_f1": variant_report.get("end_to_end", {}).get("f1"),
            "decision_accuracy": variant_report.get("decision", {}).get("accuracy"),
        }

    display(pd.DataFrame(sweep_results).T)
else:
    print("RUN_CONFIG_SWEEP is False — skip (this would overwrite data/outputs/ once per variant).")

## 12. Summary

In [ ]:
summary = {
    "Total images (\u00a73)": report.get("dataset", {}).get("total_images", "n/a"),
    "Detection F1 (\u00a74)": report.get("overall", {}).get("detection_f1", "n/a"),
    "End-to-End F1 (\u00a74)": report.get("overall", {}).get("end_to_end_f1", "n/a"),
    "Fully correct images (\u00a77)": len(bucket_fully_correct) if "bucket_fully_correct" in dir() else "n/a",
    "  ...corrected by plugin (\u00a77)": len(bucket_corrected_by_plugin) if "bucket_corrected_by_plugin" in dir() else "n/a",
    "Wrong-count images (\u00a77)": len(bucket_wrong_count) if "bucket_wrong_count" in dir() else "n/a",
    "Right-count-wrong-id images (\u00a77)": len(bucket_right_count_wrong_id) if "bucket_right_count_wrong_id" in dir() else "n/a",
    "Misidentifications in configured confusable pairs (\u00a79)": (
        int(misid["is_configured_confusable_pair"].sum()) if "misid" in dir() and not misid.empty else "n/a"
    ),
}
pd.DataFrame([summary]).T.rename(columns={0: "Value"})

---
**Quick observations (fill in after running on real data):**

- In §4, how close does the reproduced Detection/Retrieval/Fusion table match the README's headline numbers (F1 0.939, Product Count Accuracy 0.871)? Differences would point to a benchmark or config drift since the README was last updated.
- In §7, is the "corrected by plugin" bucket a large fraction of "fully correct"? That's the real-world version of the README's "+22.4% reranking gain" claim, but at the image level rather than the crop level.
- In §7's "right count, wrong identification" bucket — these are the most interesting failure mode: detection and counting both worked, but the wrong SKU was chosen. Cross-reference with §9's confusable-pair table.
- In §9, are there `gt_vs_predicted_pair` entries with high counts that are **not** in `config.rerank.confusable_pairs`? Those are strong candidates for extending the guard.
- In §10, do the slowest images correlate with more detections/crops (more plugin calls), or are some images just slow outliers worth investigating directly?

**Next notebook:** `09_error_analysis.ipynb` — deeper dive into the ~6% fine-grained identification errors the README calls out, using the buckets and confusable-pair analysis built here as the starting point.